In [1]:
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

In [2]:
data= pd.read_csv('True.csv')

In [3]:
data.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [4]:
data = data.iloc[:,:-1]

In [5]:
data.head()

,title,text,subject
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews


In [6]:
data.loc[0,'text']

'WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way among Republicans, U.S. Representative Mark Meadows, speaking on CBS’ “Face the Nation,” drew a hard line on federal spending, which lawmakers are bracing to do battle over in January. When they return from the holidays on Wednesday, lawmakers will begin trying to pass a federal budget in a fight likely to be linked to other issues, such as immigration policy, even as the November congressional election campaigns approach in which Republicans will seek to keep control of Congress. President Donald Trump and his Republicans want a big budget increase in military spending, while Democrats also want proportional increases for non-defense “discretionary” spending on programs that support educat

In [7]:
import re

In [8]:
def CleanText(text):
  text= text.lower()

  # parenthesis
  text = re.sub(r'\([^)]*\)','',text)

  # white space
  text= re.sub(r'\s+',' ',text)

  # single double quote
  text= re.sub(r'[“”’]','',text)
  text=text.strip()
  return text

In [9]:
data.iloc[:,1]= data.iloc[:,1].progress_apply(CleanText)

100%|██████████| 21417/21417 [00:04<00:00, 4379.35it/s]


In [10]:
data.shape

(21417, 3)

In [11]:
data.iloc[1,1]

'washington - transgender people will be allowed for the first time to enlist in the u.s. military starting on monday as ordered by federal courts, the pentagon said on friday, after president donald trumps administration decided not to appeal rulings that blocked his transgender ban. two federal appeals courts, one in washington and one in virginia, last week rejected the administrations request to put on hold orders by lower court judges requiring the military to begin accepting transgender recruits on jan. 1. a justice department official said the administration will not challenge those rulings. the department of defense has announced that it will be releasing an independent study of these issues in the coming weeks. so rather than litigate this interim appeal before that occurs, the administration has decided to wait for dods study and will continue to defend the presidents lawful authority in district court in the meantime, the official said, speaking on condition of anonymity. in

In [12]:
from textblob import TextBlob

In [13]:
def spellCorrect(text):
  try:
    text= TextBlob(text)
    return text.correct()
  except Exception as e:
    print(e)
    return text

In [14]:
# data.iloc[:,1]= data.iloc[:,1].progress_apply(spellCorrect)

In [15]:
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [16]:
import nltk
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [17]:
lemmatize= WordNetLemmatizer()

In [18]:
# l=['hi','hello','are','how','are','you']
# ' '.join(l)

In [19]:
def lemmatizeText(text):
  words= word_tokenize(text)
  lemmatize_words= [lemmatize.lemmatize(word) for word in words]
  return ' '.join(lemmatize_words)

In [20]:
data.iloc[:,1]= data.iloc[:,1].progress_apply(lemmatizeText)

100%|██████████| 21417/21417 [01:13<00:00, 292.59it/s]


In [21]:
data.head()

,title,text,subject
0,"As U.S. budget fight looms, Republicans flip t...",washington - the head of a conservative republ...,politicsNews
1,U.S. military to accept transgender recruits o...,washington - transgender people will be allowe...,politicsNews
2,Senior U.S. Republican senator: 'Let Mr. Muell...,washington - the special counsel investigation...,politicsNews
3,FBI Russia probe helped by Australian diplomat...,washington - trump campaign adviser george pap...,politicsNews
4,Trump wants Postal Service to charge 'much mor...,seattle/washington - president donald trump ca...,politicsNews


In [22]:
# tfidf, countVectorizer
# Word2Vec -- gensim


In [23]:
# !pip install gensim

In [24]:
from gensim.models import Word2Vec

In [25]:
model = Word2Vec(sentences=data.iloc[:,1],vector_size=10, window=5 )
# contextual -- seq2seq -- language translation

In [26]:
def ContextVector(text):
  vectors=[]
  for sent in text:
    for word in sent:
      if word in model.wv:
        vectors.append(model.wv[word])
  return vectors

In [27]:
data.iloc[:,1]= data.iloc[:,1].progress_apply(ContextVector)

100%|██████████| 21417/21417 [00:55<00:00, 385.09it/s]


In [30]:
data.iloc[1,1]

# washington - [ 2.149576  , -1.0850279 ,  0.9089034 ,  0.55297935,  0.34034497,
        #  0.7052189 , -0.07809927,  0.72094375,  2.568577  , -0.6345419 ]

[array([ 2.149576  , -1.0850279 ,  0.9089034 ,  0.55297935,  0.34034497,
         0.7052189 , -0.07809927,  0.72094375,  2.568577  , -0.6345419 ],
       dtype=float32),
 array([-0.28013578,  0.6447872 , -0.4988087 , -0.22010285,  0.43065253,
         0.86790377, -0.234548  ,  0.23063889,  0.6313273 , -0.8366173 ],
       dtype=float32),
 array([ 0.15525371, -0.7862572 , -0.31932208,  0.42571422,  0.09566152,
         0.25923264,  0.5310688 ,  0.8715865 ,  0.1248146 ,  0.7045268 ],
       dtype=float32),
 array([ 0.79599434, -0.7361688 , -0.34541732,  0.7419366 , -1.5819823 ,
         0.7210579 ,  0.17233516,  1.4063601 , -0.23722169, -1.4575572 ],
       dtype=float32),
 array([-0.41400588,  0.9725818 , -0.8906134 , -0.2890857 ,  0.3933602 ,
         1.2214833 , -0.48460275, -0.36993045,  0.13006577, -0.86497086],
       dtype=float32),
 array([ 0.69864124,  0.3127843 ,  0.45473093, -0.48900786, -0.214008  ,
         1.0936956 , -0.07382456,  0.687736  , -1.3516172 , -0.26333216],
   